# LargeGraphsJL layout demos

This notebook shows the built-in layout algorithms and both supported usage styles:

- direct layout functions
- `render(...; layout=...)` convenience calls


In [ ]:
using Pkg
Pkg.activate(isdir(joinpath(pwd(), "src")) ? pwd() : joinpath(pwd(), ".."))
Pkg.instantiate()

using LargeGraphsJL
using Random

small_nodes = [
    (; id="a", label="A", color="#2563eb", size=3.0),
    (; id="b", label="B", color="#059669", size=2.7),
    (; id="c", label="C", color="#d97706", size=2.5),
    (; id="d", label="D", color="#7c3aed", size=2.2),
    (; id="e", label="E", color="#db2777", size=2.0),
]

small_edges = [
    (; source="a", target="b", color="#94a3b8"),
    (; source="a", target="c", color="#94a3b8"),
    (; source="b", target="d", color="#94a3b8"),
    (; source="c", target="d", color="#94a3b8"),
    (; source="d", target="e", color="#94a3b8"),
]

function random_graph(node_count::Integer=5_000, edge_count::Integer=15_000; seed::Integer=7)
    rng = MersenneTwister(seed)
    nodes = Vector{NamedTuple}(undef, node_count)
    for i in 1:node_count
        nodes[i] = (
            id=string(i),
            size=1.0 + 2.0 * rand(rng),
            color=rand(rng, ("#2563eb", "#0891b2", "#059669", "#d97706")),
            label=i <= 120 ? "node-$i" : nothing,
        )
    end

    edges = NamedTuple[]
    seen = Set{Tuple{Int, Int}}()
    while length(edges) < edge_count
        source = rand(rng, 1:node_count)
        target = rand(rng, 1:node_count)
        source == target && continue
        edge = source < target ? (source, target) : (target, source)
        edge in seen && continue
        push!(seen, edge)
        push!(edges, (
            source=string(source),
            target=string(target),
            size=0.5,
            color="#cbd5e1",
        ))
    end

    nodes, edges
end


## Direct layout functions


### `random_layout`


In [ ]:
random_nodes = random_layout(small_nodes; seed=11, extent=2.0)
random_nodes

### `circular_layout`


In [ ]:
circular_nodes = circular_layout(small_nodes; radius=2.0)
circular_nodes

### `grid_layout`


In [ ]:
grid_nodes = grid_layout(small_nodes; columns=3, spacing=1.6)
grid_nodes

### `spring_layout`


In [ ]:
spring_nodes = spring_layout(small_nodes, small_edges; iterations=80, seed=5, extent=2.0)
spring_nodes

## `render(...; layout=...)` demos


### Render with `:random`


In [ ]:
render(small_nodes, small_edges; layout=:random, seed=3, height="420px")

### Render with `:circular`


In [ ]:
render(small_nodes, small_edges; layout=:circular, radius=2.2, height="420px")

### Render with `:grid`


In [ ]:
render(small_nodes, small_edges; layout=:grid, columns=3, spacing=1.5, height="420px")

### Render with `:spring`


In [ ]:
render(small_nodes, small_edges; layout=:spring, iterations=100, seed=9, extent=2.0, height="420px")

## Custom callable layout


### Render with a custom diagonal layout


In [ ]:
function diagonal_layout(nodes, edges; gap=1.0)
    [
        NodeSpec(node.id; x=index * gap, y=-index * gap, size=node.size, label=node.label, color=node.color, attributes=node.attributes)
        for (index, node) in pairs(nodes)
    ]
end

render(small_nodes, small_edges; layout=diagonal_layout, gap=0.8, height="420px")

## Large graph example


### Large graph rendered with `:random`


In [ ]:
large_nodes, large_edges = random_graph()
render(large_nodes, large_edges; layout=:random, seed=7, height="780px", hide_edges_on_move=true, label_density=0.6, max_node_size=10.0, min_node_size=1.5)

## Notes

- `:random`, `:circular`, and `:grid` are the cheaper built-in layouts.
- `:spring` is structure-aware but much more expensive on large graphs.
- For very large graphs, prefer `:random` or provide your own precomputed coordinates.
